## Desafio 3 da Trilha de Visão Computacional
### Solução: rastreamento por centróides (sem usar `model.track()`)

### Instalação e Importação da Biblioteca

In [10]:
!pip install ultralytics opencv-python torch scipy numpy

In [2]:
import ultralytics
import cv2
import torch
import numpy as np
from collections import OrderedDict
from scipy.spatial import distance as dist

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\jones\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### Implementação do rastreador por centróides

In [3]:
class CentroidTracker:
    """Rastreador simples baseado em distância entre centróides.

    Não usa nenhum modelo de movimento (como Kalman Filter) — apenas associa
    a detecção mais próxima do objeto já rastreado, frame a frame.
    """

    def __init__(self, max_disappeared=30, max_distance=60):
        # próximo ID único a ser atribuído
        self.next_object_id = 0
        # dicionário: object_id -> centróide (x, y) atual
        self.objects = OrderedDict()
        # dicionário: object_id -> nº de frames consecutivos sem ser visto
        self.disappeared = OrderedDict()
        # nº máximo de frames que um objeto pode ficar sem aparecer antes de ser removido
        self.max_disappeared = max_disappeared
        # distância máxima (em pixels) para considerar que duas detecções são o mesmo objeto
        self.max_distance = max_distance

    def register(self, centroid):
        self.objects[self.next_object_id] = centroid
        self.disappeared[self.next_object_id] = 0
        self.next_object_id += 1

    def deregister(self, object_id):
        del self.objects[object_id]
        del self.disappeared[object_id]

    def update(self, rects):
        """Atualiza o rastreador com as caixas (x1, y1, x2, y2) detectadas no frame atual."""

        # nenhuma detecção nesse frame: incrementa o contador de "sumiço" de todo mundo
        if len(rects) == 0:
            for object_id in list(self.disappeared.keys()):
                self.disappeared[object_id] += 1
                if self.disappeared[object_id] > self.max_disappeared:
                    self.deregister(object_id)
            return self.objects

        # calcula o centróide de cada caixa detectada nesse frame
        input_centroids = np.zeros((len(rects), 2), dtype="int")
        for i, (start_x, start_y, end_x, end_y) in enumerate(rects):
            c_x = int((start_x + end_x) / 2.0)
            c_y = int((start_y + end_y) / 2.0)
            input_centroids[i] = (c_x, c_y)

        # se ainda não há nenhum objeto sendo rastreado, registra todos como novos
        if len(self.objects) == 0:
            for i in range(len(input_centroids)):
                self.register(input_centroids[i])
        else:
            object_ids = list(self.objects.keys())
            object_centroids = list(self.objects.values())

            # matriz de distâncias entre cada objeto já rastreado e cada nova detecção
            D = dist.cdist(np.array(object_centroids), input_centroids)

            # ordena as linhas pela menor distância encontrada em cada uma
            rows = D.min(axis=1).argsort()
            # para cada linha, pega a coluna (nova detecção) com menor distância
            cols = D.argmin(axis=1)[rows]

            used_rows = set()
            used_cols = set()

            for (row, col) in zip(rows, cols):
                if row in used_rows or col in used_cols:
                    continue
                # distância grande demais: provavelmente não é o mesmo objeto
                if D[row, col] > self.max_distance:
                    continue

                object_id = object_ids[row]
                self.objects[object_id] = input_centroids[col]
                self.disappeared[object_id] = 0

                used_rows.add(row)
                used_cols.add(col)

            unused_rows = set(range(D.shape[0])).difference(used_rows)
            unused_cols = set(range(D.shape[1])).difference(used_cols)

            # objetos que já conhecíamos e não casaram com nenhuma detecção nova
            if D.shape[0] >= D.shape[1]:
                for row in unused_rows:
                    object_id = object_ids[row]
                    self.disappeared[object_id] += 1
                    if self.disappeared[object_id] > self.max_disappeared:
                        self.deregister(object_id)
            # detecções novas que não casaram com nenhum objeto conhecido
            else:
                for col in unused_cols:
                    self.register(input_centroids[col])

        return self.objects

### Processar dados

In [4]:
# Baixa o modelo
model = ultralytics.YOLO("yolo11n.pt")

# Joga o modelo para o dispositivo de computação
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Cria o rastreador por centróides
tracker = CentroidTracker(max_disappeared=30, max_distance=60)

### Detecção + rastreamento frame a frame

In [5]:
cap = cv2.VideoCapture("fruit-and-vegetable-detection.mp4")

fps = cap.get(cv2.CAP_PROP_FPS) or 20
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"FPS: {fps} | Resolução: {frame_width}x{frame_height}")

FPS: 59.94005994005994 | Resolução: 960x540


In [6]:
# guarda, para cada frame, as caixas detectadas e os objetos rastreados
# (útil para inspecionar os resultados antes de gerar o vídeo, similar ao "validar resultados" do gabarito)
all_frames = []
all_detections = []
all_tracked_objects = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # detecção pura do YOLO, sem nenhum rastreador embutido
    result = model.predict(frame, conf=0.1, iou=0.7, verbose=False)[0]

    rects = []
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        rects.append((int(x1), int(y1), int(x2), int(y2)))

    # atualiza o rastreador com as caixas desse frame
    objects = tracker.update(rects)

    all_frames.append(frame)
    all_detections.append(rects)
    all_tracked_objects.append(dict(objects))

cap.release()
print(f"Total de frames processados: {len(all_frames)}")

Total de frames processados: 3633


### Validar resultados

In [7]:
# validar resultados: quantos objetos distintos foram rastreados e quantas detecções por frame
for i, (rects, objects) in enumerate(zip(all_detections, all_tracked_objects)):
    print(f"Frame {i}: {len(rects)} detecções | {len(objects)} objetos rastreados | IDs: {list(objects.keys())}")

Frame 0: 0 detecções | 0 objetos rastreados | IDs: []
Frame 1: 0 detecções | 0 objetos rastreados | IDs: []
Frame 2: 0 detecções | 0 objetos rastreados | IDs: []
Frame 3: 0 detecções | 0 objetos rastreados | IDs: []
Frame 4: 0 detecções | 0 objetos rastreados | IDs: []
Frame 5: 0 detecções | 0 objetos rastreados | IDs: []
Frame 6: 0 detecções | 0 objetos rastreados | IDs: []
Frame 7: 0 detecções | 0 objetos rastreados | IDs: []
Frame 8: 0 detecções | 0 objetos rastreados | IDs: []
Frame 9: 0 detecções | 0 objetos rastreados | IDs: []
Frame 10: 0 detecções | 0 objetos rastreados | IDs: []
Frame 11: 0 detecções | 0 objetos rastreados | IDs: []
Frame 12: 0 detecções | 0 objetos rastreados | IDs: []
Frame 13: 0 detecções | 0 objetos rastreados | IDs: []
Frame 14: 0 detecções | 0 objetos rastreados | IDs: []
Frame 15: 0 detecções | 0 objetos rastreados | IDs: []
Frame 16: 0 detecções | 0 objetos rastreados | IDs: []
Frame 17: 0 detecções | 0 objetos rastreados | IDs: []
Frame 18: 0 detecçõe

### Salvar Vídeo

In [8]:
# Cria o escritor do vídeo de output
# Você pode pesquisar para salvar em outros formatos
writer = cv2.VideoWriter(
    "output_centroid_tracker.avi",
    cv2.VideoWriter_fourcc(*"XVID"),
    fps,
    (frame_width, frame_height),
)

for frame, rects, objects in zip(all_frames, all_detections, all_tracked_objects):
    annotated = frame.copy()

    # desenha as caixas detectadas pelo YOLO
    for (x1, y1, x2, y2) in rects:
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # desenha o ID e o centróide atribuídos pelo nosso rastreador
    for object_id, centroid in objects.items():
        text = f"ID {object_id}"
        cv2.putText(
            annotated, text, (centroid[0] - 10, centroid[1] - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2,
        )
        cv2.circle(annotated, (int(centroid[0]), int(centroid[1])), 4, (0, 0, 255), -1)

    writer.write(annotated)

writer.release()
print("Vídeo salvo em output_centroid_tracker.avi")

Vídeo salvo em output_centroid_tracker.avi


### Referência

- https://docs.ultralytics.com/modes/track/
- https://docs.ultralytics.com/modes/predict/
- Algoritmo de Centroid Tracking: https://pyimagesearch.com/2018/07/23/simple-object-tracking-with-opencv/